# Week 48, Day 1: Signal detection theory: sensitivity and bias

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w048/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Payoffs move the criterion

Write sdt(H, F) returning d' and c. Apply it to the predict question's two conditions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

def sdt(H, F):
    ...

results = {"neutral": sdt(0.69, 0.16), "liberal": sdt(0.84, 0.31)}
for k, (d, c) in results.items():
    print(f"{k:8s}: d' = {d:.2f}, c = {c:+.2f}")

In [ ]:
import numpy as np
def test_same_dprime():
    "d' is the same (within 0.05) under both payoffs"
    assert abs(results["neutral"][0] - results["liberal"][0]) < 0.05
def test_criterion():
    "the liberal payoff lowers the criterion by more than 0.4"
    assert results["neutral"][1] - results["liberal"][1] > 0.4

test_same_dprime()
test_criterion()
print("All checks passed")

### Exercise: Ratings, ROC and the z-ROC slope

Simulate 2,000 noise and 2,000 signal trials (d' = 1.5) with rating boundaries at -0.5, 0, 0.5, 1, 1.5. For each boundary compute H and F as the fraction of ratings above it, fit a line to (z(F), z(H)), and compute the area under the ROC with the trapezium rule (adding the corners 0,0 and 1,1).

In [ ]:
r = np.random.default_rng(1954)
bounds = np.array([-0.5, 0.0, 0.5, 1.0, 1.5])

def roc(sd_signal, n=2000, d=1.5):
    noise = r.normal(0, 1, n); signal = r.normal(d, sd_signal, n)
    H = ...
    F = ...
    return H, F

slope, auc = {}, {}
for name, sd in [("equal", 1.0), ("unequal", 1.5)]:
    H, F = roc(sd)
    slope[name] = np.polyfit(norm.ppf(F), norm.ppf(H), 1)[0]
    Fs, Hs = np.r_[0, F[::-1], 1], np.r_[0, H[::-1], 1]
    auc[name] = np.trapz(Hs, Fs) if hasattr(np, "trapz") else np.trapezoid(Hs, Fs)
    plt.plot(Fs, Hs, "o-", label=name)
    print(f"{name:8s}: z-ROC slope {slope[name]:.2f}, area under ROC {auc[name]:.3f}")
plt.plot([0, 1], [0, 1], "k:"); plt.xlabel("false-alarm rate"); plt.ylabel("hit rate"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_equal():
    "for the equal-variance observer the z-ROC slope is within 0.1 of 1"
    assert abs(slope["equal"] - 1) < 0.1
def test_unequal():
    "for the unequal-variance observer (signal sd 1.5) the slope is near 1/1.5"
    assert abs(slope["unequal"] - 1 / 1.5) < 0.1
def test_auc():
    "the equal-variance area under the ROC is within 0.04 of Phi(d'/sqrt 2)"
    assert abs(auc["equal"] - norm.cdf(1.5 / np.sqrt(2))) < 0.04

test_equal()
test_unequal()
test_auc()
print("All checks passed")

### Exercise: 2AFC from first principles

For d' of 0.5, 1, 1.5 and 2, simulate 20,000 2AFC trials and compare the proportion correct with Phi(d'/sqrt 2).

In [ ]:
pc_sim = {}
for d in [0.5, 1.0, 1.5, 2.0]:
    s, n_ = r.normal(d, 1, 20000), r.normal(0, 1, 20000)
    pc_sim[d] = ...
    print(f"d' = {d}: simulated {pc_sim[d]:.3f}, formula {norm.cdf(d / np.sqrt(2)):.3f}")

In [ ]:
import numpy as np
def test_match():
    "simulated 2AFC accuracy matches Phi(d'/sqrt 2) within 0.015 for every d'"
    assert all(abs(pc_sim[d] - norm.cdf(d / np.sqrt(2))) < 0.015 for d in pc_sim)

test_match()
print("All checks passed")